# Part 1 — Does DoRA's magnitude/direction claim transfer to a VLM's vision encoder?

**Paper:** DoRA: Weight-Decomposed Low-Rank Adaptation (Liu et al., 2024).

**Central claim.** Decomposing a weight into *magnitude* and *direction* and applying LoRA only to the direction lets the
update behave like full fine-tuning (FT): in FT, magnitude and direction updates are **negatively** correlated across
layers, whereas plain LoRA's are **positively** correlated (paper Fig. 2, §4.1). DoRA restores the FT-like negative pattern
and improves accuracy at equal rank.

**Gap.** The paper's analysis was done on VL-BART, whose visual backbone is frozen, so the pattern was never measured
*inside a vision transformer*.

**Hypothesis.** When fine-tuning SmolVLM-256M on VQAv2:
- **H1 (transfer):** in the language decoder, FT and DoRA show a negative ΔM–ΔD correlation and LoRA a positive one.
- **H2 (vision-specific):** the same sign pattern holds inside the SigLIP vision encoder (or differs, which we test and report).
- **H3:** DoRA ≥ LoRA in VQA accuracy at equal rank.

**Claim-tied metric.** For every tracked weight `W` (q_proj, v_proj in each vision / text layer), at several training checkpoints:

$$\Delta M_t = \tfrac{1}{k}\sum_n |m^n_t - m^n_0|, \qquad \Delta D_t = \tfrac{1}{k}\sum_n \big(1-\cos(W^n_t, W^n_0)\big)$$

with `n` running over output rows (the norm DoRA's magnitude vector parameterises). We report the across-layer Pearson
correlation between ΔD and ΔM for each method × tower × projection. Accuracy is secondary.

In [ ]:
!pip -q install peft datasets accelerate scipy
import transformers, peft, torch
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Config (fixed seed for reproducibility)

In [ ]:
import os, re, gc, json, math, time, random
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoProcessor, TrainingArguments, Trainer, TrainerCallback, set_seed
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM
from peft import LoraConfig, get_peft_model

SEED      = 42
MODEL_ID  = "HuggingFaceTB/SmolVLM-256M-Instruct"
DATASET   = "merve/vqav2-small"          # VQAv2 validation subset (parquet, ~21k QA pairs)
N_TRAIN   = 3000
N_TEST    = 500
EPOCHS    = 2
BATCH     = 8
RANK, ALPHA, DROPOUT = 8, 16, 0.05
LR = {"ft": 1e-5, "lora": 2e-4, "dora": 2e-4}
N_RECORD  = 8                            # number of checkpoints at which dM/dD are measured
PROMPT    = "{q}\nAnswer with a single word or short phrase."
TARGET_REGEX = r".*(vision_model|text_model).*\.(q_proj|v_proj)"   # same projections as the paper (query, value)

set_seed(SEED); random.seed(SEED); np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/EN4554_A01/part1"   # checkpointed to Drive so a disconnect is not fatal
except ImportError:
    OUT_DIR = "./outputs/part1"
os.makedirs(OUT_DIR, exist_ok=True)
print("Outputs ->", OUT_DIR)

## 2. Data — VQAv2 subset, split by image so no test image is seen in training

In [ ]:
raw = load_dataset(DATASET, split="validation")
print(raw)

rng = random.Random(SEED)
if "image_id" in raw.column_names:
    img_ids = raw["image_id"]
    uniq = sorted(set(img_ids)); rng.shuffle(uniq)
    test_imgs = set(uniq[: len(uniq) // 10])
    train_idx = [i for i, x in enumerate(img_ids) if x not in test_imgs]
    test_idx  = [i for i, x in enumerate(img_ids) if x in test_imgs]
else:
    idx = list(range(len(raw))); rng.shuffle(idx)
    test_idx, train_idx = idx[: len(idx) // 10], idx[len(idx) // 10:]
rng.shuffle(train_idx); rng.shuffle(test_idx)
train_ds = raw.select(train_idx[:N_TRAIN])
test_ds  = raw.select(test_idx[:N_TEST])
print(f"train={len(train_ds)}  test={len(test_ds)}  (split: {DATASET}/validation, seed={SEED})")

def gt_answers(e):
    a = e.get("answers")
    if a:
        return [x["answer"] if isinstance(x, dict) else x for x in a]
    return [e["multiple_choice_answer"]]

ex = train_ds[0]; print(ex["question"], "->", ex["multiple_choice_answer"], "| all:", gt_answers(ex)[:10])

## 3. Processor, chat formatting and an assistant-only loss collator

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.do_image_splitting = False    # one 512px tile -> 64 image tokens (keeps Colab runs short)
tok = processor.tokenizer
ASSIST_IDS = tok.encode("Assistant:", add_special_tokens=False)

def build_msgs(q, a=None):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT.format(q=q)}]}]
    if a is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": a}]})
    return msgs

def find_last(seq, sub):
    for i in range(len(seq) - len(sub), -1, -1):
        if seq[i:i + len(sub)] == sub:
            return i
    return -1

def train_collate(examples):
    texts  = [processor.apply_chat_template(build_msgs(e["question"], e["multiple_choice_answer"])) for e in examples]
    images = [[e["image"].convert("RGB")] for e in examples]
    tok.padding_side = "right"
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    for i, ids in enumerate(batch["input_ids"].tolist()):
        j = find_last(ids, ASSIST_IDS)
        labels[i, : (j + len(ASSIST_IDS)) if j >= 0 else len(ids)] = -100   # loss on the assistant turn only
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch

# sanity check: the only supervised tokens should be the answer
b = train_collate([train_ds[0], train_ds[1]])
for i in range(2):
    keep = b["labels"][i][b["labels"][i] != -100]
    print("supervised:", repr(tok.decode(keep)), "| seq len", b["input_ids"].shape[1])

## 4. VQA evaluation (VQAv2 soft accuracy: min(#annotators agreeing / 3, 1))

In [ ]:
def normalize(s):
    s = s.lower().strip().split("\n")[0]
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

@torch.no_grad()
def evaluate(model, ds, bs=32, max_new_tokens=10):
    model.eval(); tok.padding_side = "left"
    rows = []
    for s in tqdm(range(0, len(ds), bs), desc="eval"):
        exs = ds.select(range(s, min(s + bs, len(ds))))
        texts  = [processor.apply_chat_template(build_msgs(e["question"]), add_generation_prompt=True) for e in exs]
        images = [[e["image"].convert("RGB")] for e in exs]
        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(model.device)
        with torch.autocast("cuda", dtype=torch.float16):
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gens = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for e, g in zip(exs, gens):
            p = normalize(g); gts = [normalize(a) for a in gt_answers(e)]
            rows.append(dict(question=e["question"], pred=p, gt=e["multiple_choice_answer"],
                             acc=min(sum(a == p for a in gts) / 3, 1.0)))
    return float(np.mean([r["acc"] for r in rows])), rows

def load_model():
    return AutoVLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).float().to("cuda")  # fp32 master weights + fp16 autocast

## 5. Magnitude / direction tracker

For LoRA and DoRA the *effective* weight is rebuilt from the adapter (`W0 + s·BA`, then DoRA's `m · V/‖V‖`), so all three
methods are measured the same way. Rows of the PyTorch `(out, in)` weight are the vectors DoRA's magnitude vector scales.

In [ ]:
TARGET_RE = re.compile(TARGET_REGEX)

def tracked_modules(model):
    out = {}
    for name, mod in model.named_modules():
        clean = name.replace("base_model.model.", "")
        if TARGET_RE.fullmatch(clean) and (isinstance(mod, nn.Linear) or hasattr(mod, "base_layer")):
            out[clean] = mod
    return out

@torch.no_grad()
def effective_weight(mod, adapter="default"):
    if hasattr(mod, "base_layer"):
        W0 = mod.base_layer.weight.double()
        A, B = mod.lora_A[adapter].weight.double(), mod.lora_B[adapter].weight.double()
        W = W0 + mod.scaling[adapter] * (B @ A)
        if mod.use_dora.get(adapter, False):
            m = mod.lora_magnitude_vector[adapter].weight.double()
            W = (m / W.norm(dim=1)).unsqueeze(1) * W
        return W
    return mod.weight.double()

class MagDirTracker(TrainerCallback):
    def __init__(self, model, record_steps):
        self.mods, self.steps, self.records = tracked_modules(model), set(record_steps), []
        self.W0 = {n: effective_weight(m).clone() for n, m in self.mods.items()}
        self.m0 = {n: W.norm(dim=1) for n, W in self.W0.items()}
        print(f"tracking {len(self.mods)} matrices "
              f"({sum('vision_model' in n for n in self.mods)} vision, {sum('text_model' in n for n in self.mods)} text)")

    @torch.no_grad()
    def measure(self, step):
        for n, mod in self.mods.items():
            W = effective_weight(mod)
            self.records.append(dict(
                step=step, module=n,
                tower="vision" if "vision_model" in n else "text",
                proj="q" if n.endswith("q_proj") else "v",
                layer=int(re.search(r"layers\.(\d+)\.", n).group(1)),
                dM=(W.norm(dim=1) - self.m0[n]).abs().mean().item(),
                dD=(1 - F.cosine_similarity(W, self.W0[n], dim=1)).mean().item()))

    def on_step_end(self, args, state, control, **kw):
        if state.global_step in self.steps:
            self.measure(state.global_step)

## 6. Training runs: full FT, LoRA, DoRA (same data, seed, rank and target modules). Each run is resumable from Drive.

In [ ]:
def run(method):
    path = f"{OUT_DIR}/{method}"; res_file = f"{path}/results.json"
    if os.path.exists(res_file):
        print(f"[{method}] loading cached results"); return json.load(open(res_file))
    os.makedirs(path, exist_ok=True); set_seed(SEED)

    model = load_model()
    if method != "ft":
        cfg = LoraConfig(r=RANK, lora_alpha=ALPHA, lora_dropout=DROPOUT, target_modules=TARGET_REGEX,
                         use_dora=(method == "dora"), bias="none")
        model = get_peft_model(model, cfg)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"[{method}] trainable {n_train/1e6:.3f}M / {n_total/1e6:.1f}M ({100*n_train/n_total:.3f}%)")

    total_steps = math.ceil(N_TRAIN / BATCH) * EPOCHS
    record_steps = sorted(set(np.linspace(total_steps / N_RECORD, total_steps, N_RECORD).round().astype(int).tolist()))
    tracker = MagDirTracker(model, record_steps)

    args = TrainingArguments(
        output_dir=f"{path}/tmp", num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
        learning_rate=LR[method], lr_scheduler_type="linear", warmup_steps=int(0.05 * total_steps),
        weight_decay=0.0, max_grad_norm=1.0, fp16=True, logging_steps=10, save_strategy="no",
        report_to="none", remove_unused_columns=False, label_names=["labels"],
        dataloader_num_workers=2, seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=train_collate, callbacks=[tracker])

    t0 = time.time(); trainer.train(); train_min = (time.time() - t0) / 60
    acc, preds = evaluate(model, test_ds)
    print(f"[{method}] VQA acc = {100*acc:.2f}%  ({train_min:.1f} min)")

    res = dict(method=method, acc=acc, trainable=n_train, total=n_total, train_min=train_min, lr=LR[method],
               records=tracker.records, log_history=trainer.state.log_history, preds=preds)
    json.dump(res, open(res_file, "w"))
    if method != "ft":
        model.save_pretrained(f"{path}/adapter")
    del model, trainer, tracker; gc.collect(); torch.cuda.empty_cache()
    return res

### 6a. Baseline (untouched model)

In [ ]:
base_file = f"{OUT_DIR}/baseline.json"
if os.path.exists(base_file):
    baseline = json.load(open(base_file))
else:
    m = load_model(); acc, preds = evaluate(m, test_ds)
    baseline = dict(method="baseline", acc=acc, preds=preds); json.dump(baseline, open(base_file, "w"))
    del m; gc.collect(); torch.cuda.empty_cache()
print(f"baseline VQA acc = {100*baseline['acc']:.2f}%")

### 6b. Fine-tuning runs (~ a few minutes each on a T4; re-running a cell picks up cached results)

In [ ]:
results = {m: run(m) for m in ["ft", "lora", "dora"]}

## 7. Results

In [ ]:
acc_tbl = pd.DataFrame([dict(method="baseline (zero-shot)", vqa_acc=100*baseline["acc"])] + [
    dict(method=m.upper(), vqa_acc=100*r["acc"], trainable_M=r["trainable"]/1e6,
         trainable_pct=100*r["trainable"]/r["total"], train_min=r["train_min"]) for m, r in results.items()])
display(acc_tbl.round(3))

### 7a. The claim-tied metric: ΔD vs ΔM

Primary statistic: at each checkpoint, the Pearson *r* between ΔD and ΔM **across layers**, averaged over checkpoints.
We also report the pooled *r* and regression slope over all (layer, checkpoint) points, which is how Fig. 2 is visualised.

In [ ]:
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

df = pd.concat([pd.DataFrame(r["records"]).assign(method=m) for m, r in results.items()], ignore_index=True)

def safe_r(x, y):
    return pearsonr(x, y)[0] if len(x) > 2 and np.std(x) > 0 and np.std(y) > 0 else np.nan

rows = []
for (m, tw, pj), g in df.groupby(["method", "tower", "proj"]):
    per_ckpt = [safe_r(gg.dD, gg.dM) for _, gg in g.groupby("step")]
    r_pool, p_pool = pearsonr(g.dD, g.dM)
    rows.append(dict(method=m.upper(), tower=tw, proj=pj, n_layers=g.layer.nunique(),
                     r_across_layers=np.nanmean(per_ckpt), r_pooled=r_pool, p_pooled=p_pool,
                     slope=np.polyfit(g.dD, g.dM, 1)[0]))
corr_tbl = pd.DataFrame(rows).sort_values(["tower", "proj", "method"])
display(corr_tbl.round(4))
corr_tbl.to_csv(f"{OUT_DIR}/corr_table.csv", index=False)

In [ ]:
methods, panels = ["ft", "lora", "dora"], [("vision", "q"), ("vision", "v"), ("text", "q"), ("text", "v")]
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i, m in enumerate(methods):
    for j, (tw, pj) in enumerate(panels):
        ax, g = axes[i, j], df[(df.method == m) & (df.tower == tw) & (df.proj == pj)]
        sc = ax.scatter(g.dD, g.dM, c=g.step, cmap="viridis", s=14)
        k = np.polyfit(g.dD, g.dM, 1); xs = np.linspace(g.dD.min(), g.dD.max(), 50)
        ax.plot(xs, np.polyval(k, xs), "r--", lw=1)
        ax.set_title(f"{m.upper()} | {tw} {pj}_proj | r={pearsonr(g.dD, g.dM)[0]:+.2f}")
        ax.set_xlabel("ΔD (1 − cos)"); ax.set_ylabel("ΔM")
        ax.ticklabel_format(style="sci", scilimits=(-2, 2))
fig.colorbar(sc, ax=axes, label="training step", shrink=0.6)
plt.savefig(f"{OUT_DIR}/dM_vs_dD.png", dpi=150, bbox_inches="tight"); plt.show()

### 7b. Bonus (vision-specific): where in depth does each tower change magnitude vs direction?

Final-checkpoint ΔD and ΔM against relative depth, vision encoder vs language decoder. If the vision encoder adapts
mostly through direction in early (low-level) layers and the LM through magnitude, that is a vision-specific finding
the NLP paper could not show.

In [ ]:
last = df[df.step == df.groupby("method").step.transform("max")].copy()
last["depth"] = last.layer / last.groupby(["method", "tower"]).layer.transform("max")
fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharex=True)
for j, m in enumerate(methods):
    for i, metric in enumerate(["dD", "dM"]):
        ax = axes[i, j]
        for (tw, pj), g in last[last.method == m].groupby(["tower", "proj"]):
            g = g.sort_values("depth"); ax.plot(g.depth, g[metric], marker="o", ms=3, label=f"{tw}-{pj}")
        ax.set_title(f"{m.upper()} — {metric} at final step"); ax.set_xlabel("relative layer depth")
        ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2)); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/depth_profile.png", dpi=150, bbox_inches="tight"); plt.show()

# ratio of direction change to magnitude change (scale-free summary per tower)
ratio = (last.groupby(["method", "tower"])[["dD", "dM"]].mean().assign(dD_over_dM=lambda t: t.dD / t.dM))
display(ratio)

### 7c. Training loss curves

In [ ]:
plt.figure(figsize=(7, 4))
for m, r in results.items():
    lh = [x for x in r["log_history"] if "loss" in x]
    plt.plot([x["step"] for x in lh], [x["loss"] for x in lh], label=m.upper())
plt.xlabel("step"); plt.ylabel("train loss"); plt.legend(); plt.grid(alpha=.3)
plt.savefig(f"{OUT_DIR}/loss.png", dpi=150, bbox_inches="tight"); plt.show()

## 8. Reading the result

- **H1 holds** if `r_across_layers` is negative for FT and DoRA and positive for LoRA in the `text` rows.
- **H2 holds** if the same sign pattern appears in the `vision` rows. A mismatch between towers is a finding in itself:
  it means the decomposition argument doesn't carry over unchanged to a frozen-then-unfrozen ViT.
- **H3** is answered by the accuracy table (DoRA vs LoRA at the same rank and trainable-parameter budget).

Report: seed=42, the data split sizes printed in §2, rank / α / LR from §1, and the trainable-parameter column.